# RoBERTa FOMC Sentiment Classification

Fine-tunes `roberta-base` for Dovish, Hawkish, and Neutral classification. Run the cells from top to bottom.


In [2]:
"""
COMP9444 FOMC Sentiment Classification

Model:  roberta-base → 3-class classification head → full fine-tuning
Labels: 0 = Dovish, 1 = Hawkish, 2 = Neutral

Outputs per seed
----------------
results/roberta_base_combined/seed_<SEED>/
    config.json
    metrics.json
    predictions.csv
    classification_report.csv
    training_history.csv
    learning_curve.png
    confusion_matrix_raw.csv
    confusion_matrix_normalized.csv
    confusion_matrix_normalized.png

results/roberta_base_combined/
    aggregate_metrics.json

Extension (Combined-S, sentence-split):
results/roberta_base_combined_split/  (same layout)
"""

from __future__ import annotations

import argparse
import csv
import json
import platform
import re
import time
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


## Paths


In [3]:
WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / "9444").is_dir():
    ROOT = WORKING_DIRECTORY / "9444"
elif WORKING_DIRECTORY.name.lower() == "models":
    ROOT = WORKING_DIRECTORY.parent
elif WORKING_DIRECTORY.name == "9444":
    ROOT = WORKING_DIRECTORY
else:
    raise FileNotFoundError(
        "Run this notebook from the project root, 9444, or 9444/models directory."
    )
DATA_ZIP = ROOT / "FOMC_dataset_checkpoint.zip"
RESULTS_ROOT = ROOT / "results"

SEEDS = ["5768", "78516", "944601"]
LABEL_NAMES = ["Dovish", "Hawkish", "Neutral"]
CHECKPOINT = "roberta-base"


## Shared training settings (aligned with BERT / FinBERT members)


In [4]:
MAX_LENGTH = 256
MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
DROPOUT = 0.1

# Hyperparameter grid - tune on seed 5768, freeze, run remaining seeds
HP_GRID = [
    {"lr": 1e-5, "batch_size": 16},
    {"lr": 2e-5, "batch_size": 16},
    {"lr": 5e-5, "batch_size": 16},
    {"lr": 1e-5, "batch_size": 8},
    {"lr": 2e-5, "batch_size": 8},
    {"lr": 5e-5, "batch_size": 8},
]


## Data loading (mirrors fomc_dataset_analysis.py, pure-stdlib xlsx reader)


In [5]:
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


def _column_index(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    idx = 0
    for ch in letters:
        idx = idx * 26 + ord(ch) - ord("A") + 1
    return idx - 1


def _cell_text(cell, shared_strings: list[str]) -> str:
    cell_type = cell.attrib.get("t")
    if cell_type == "inlineStr":
        return "".join(t.text or "" for t in cell.iter(NS + "t"))
    value = cell.find(NS + "v")
    if value is None:
        return ""
    if cell_type == "s":
        return shared_strings[int(value.text)]
    return value.text or ""


def _read_xlsx_bytes(data: bytes) -> list[dict]:
    with ZipFile(BytesIO(data)) as wb:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in wb.namelist():
            root = ET.fromstring(wb.read("xl/sharedStrings.xml"))
            for item in root:
                shared_strings.append(
                    "".join(t.text or "" for t in item.iter(NS + "t"))
                )
        sheet = ET.fromstring(wb.read("xl/worksheets/sheet1.xml"))
        rows: list[list[str]] = []
        for row in sheet.find(NS + "sheetData"):
            values: list[str] = []
            for cell in row:
                idx = _column_index(cell.attrib["r"])
                while len(values) <= idx:
                    values.append("")
                values[idx] = _cell_text(cell, shared_strings)
            rows.append(values)
        headers = rows[0]
        records = []
        for row_vals in rows[1:]:
            rec = {
                headers[i]: row_vals[i] if i < len(row_vals) else ""
                for i in range(len(headers))
            }
            rec["index"] = int(rec["index"])
            rec["year"] = int(rec["year"])
            rec["label"] = int(rec["label"])
            records.append(rec)
    return records


def load_seed(seed: str) -> tuple[list[dict], list[dict]]:
    """Return (train_records, test_records) for a given seed string."""
    with ZipFile(DATA_ZIP) as archive:
        train_path = f"FOMC_dataset_checkpoint/lab-manual-combine-train-{seed}.xlsx"
        test_path = f"FOMC_dataset_checkpoint/lab-manual-combine-test-{seed}.xlsx"
        train_records = _read_xlsx_bytes(archive.read(train_path))
        test_records = _read_xlsx_bytes(archive.read(test_path))
    return train_records, test_records


## Sentence splitting (Extension: Combined-S)


In [6]:
_SPLIT_PATTERN = re.compile(r"(?<=[.!?])\s+")


def split_into_sentences(text: str) -> list[str]:
    """Naïve sentence splitter - good enough for FOMC prose."""
    parts = _SPLIT_PATTERN.split(text.strip())
    return [p.strip() for p in parts if p.strip()]


def expand_to_sentences(records: list[dict]) -> list[dict]:
    """
    For the Combined-S extension: expand each record into one record per
    sentence, keeping the parent label and a sub-index.
    """
    expanded = []
    for rec in records:
        sentences = split_into_sentences(rec["sentence"])
        if not sentences:
            sentences = [rec["sentence"]]
        for sub_idx, sent in enumerate(sentences):
            new_rec = dict(rec)
            new_rec["sentence"] = sent
            new_rec["parent_index"] = rec["index"]
            new_rec["sub_index"] = sub_idx
            # Assign a unique index
            new_rec["index"] = rec["index"] * 1000 + sub_idx
            expanded.append(new_rec)
    return expanded


## Normalise sentence text (shared pre-processing rule §2.3)


In [7]:
def normalise(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


## PyTorch Dataset


In [30]:
class FOMCDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        tokenizer,
        max_length: int = MAX_LENGTH,
    ):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        encoding = self.tokenizer(
            normalise(rec["sentence"]),
            max_length=self.max_length,
            truncation=True,
            padding=False,
            return_tensors=None,
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "label": rec["label"],
        }


def collate_fn(batch):
    """Dynamically pad and return only tensors consumed by the model."""
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [1] * pad_len)  # RoBERTa pad_token_id = 1
        attention_masks.append(item["attention_mask"] + [0] * pad_len)
        labels.append(item["label"])

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## Training helpers


In [21]:
from tqdm.auto import tqdm


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def train_one_epoch(model, loader, optimiser, scheduler, device):
    model.train()
    total_loss = 0.0
    total_training_batches = max(len(loader) * MAX_EPOCHS, 1)
    completed_batches = min(max(scheduler.last_epoch, 0), total_training_batches)
    progress = tqdm(
        total=total_training_batches,
        initial=completed_batches,
        desc="Overall training",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )
    for batch_number, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimiser.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = output.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()
        total_loss += loss.item()
        progress.update()
        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            epoch_avg=f"{total_loss / batch_number:.4f}",
        )
    progress.close()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += output.loss.item()

        probs = torch.softmax(output.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    wf1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, wf1, all_preds, all_labels, all_probs

## Result saving


In [10]:
def save_config(out_dir: Path, cfg: dict):
    with open(out_dir / "config.json", "w") as f:
        json.dump(cfg, f, indent=2)


def save_metrics(out_dir: Path, metrics: dict):
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)


def save_predictions(
    out_dir: Path,
    records: list[dict],
    preds: list[int],
    probs: list[list[float]],
    seed: str,
    model_name: str,
):
    path = out_dir / "predictions.csv"
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "sample_id", "year", "sentence", "true_label",
                "predicted_label", "prob_dovish", "prob_hawkish",
                "prob_neutral", "correct", "seed", "model_name",
            ],
        )
        writer.writeheader()
        for rec, pred, prob in zip(records, preds, probs):
            writer.writerow({
                "sample_id": rec["index"],
                "year": rec["year"],
                "sentence": rec["sentence"],
                "true_label": rec["label"],
                "predicted_label": pred,
                "prob_dovish": f"{prob[0]:.6f}",
                "prob_hawkish": f"{prob[1]:.6f}",
                "prob_neutral": f"{prob[2]:.6f}",
                "correct": int(rec["label"] == pred),
                "seed": seed,
                "model_name": model_name,
            })


def save_classification_report(out_dir: Path, true_labels, pred_labels):
    report = classification_report(
        true_labels, pred_labels,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
    with open(out_dir / "classification_report.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class", "precision", "recall", "f1-score", "support"])
        for cls in LABEL_NAMES:
            row = report[cls]
            writer.writerow([cls, row["precision"], row["recall"], row["f1-score"], row["support"]])
        for avg_key in ["macro avg", "weighted avg"]:
            row = report[avg_key]
            writer.writerow([avg_key, row["precision"], row["recall"], row["f1-score"], row["support"]])


def save_training_history(out_dir: Path, history: list[dict]):
    if not history:
        return
    fields = list(history[0].keys())
    with open(out_dir / "training_history.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(history)


def save_learning_curve(out_dir: Path, history: list[dict], best_epoch: int):
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    val_wf1 = [h["val_weighted_f1"] for h in history]

    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.plot(epochs, train_loss, label="Train Loss", color="tab:blue")
    ax1.plot(epochs, val_loss, label="Val Loss", color="tab:orange")
    ax1.axvline(x=best_epoch, color="gray", linestyle="--", label=f"Best epoch ({best_epoch})")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Validation Weighted F1")
    ax2.plot(epochs, val_wf1, label="Val Weighted F1", color="tab:green")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")

    plt.title("Learning Curve")
    plt.tight_layout()
    fig.savefig(out_dir / "learning_curve.png", dpi=150)
    plt.close(fig)


def save_confusion_matrices(out_dir: Path, true_labels, pred_labels):
    cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2])

    # Raw
    with open(out_dir / "confusion_matrix_raw.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([""] + LABEL_NAMES)
        for i, row in enumerate(cm):
            writer.writerow([LABEL_NAMES[i]] + list(row))

    # Normalised (by true class)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

    with open(out_dir / "confusion_matrix_normalized.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([""] + LABEL_NAMES)
        for i, row in enumerate(cm_norm):
            writer.writerow([LABEL_NAMES[i]] + [f"{v:.4f}" for v in row])

    # Plot
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(LABEL_NAMES)
    ax.set_yticklabels(LABEL_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Normalised)")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm_norm[i, j]:.1%}", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    plt.tight_layout()
    fig.savefig(out_dir / "confusion_matrix_normalized.png", dpi=150)
    plt.close(fig)


## Core training routine for one seed


In [11]:
def run_seed(
    seed: str,
    out_dir: Path,
    lr: float,
    batch_size: int,
    model_name: str,
    use_sentence_split: bool = False,
    device: torch.device | None = None,
):
    if device is None:
        device = get_device()

    int_seed = int(seed)
    torch.manual_seed(int_seed)
    np.random.seed(int_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int_seed)

    print(f"\n{'=' * 60}")
    print(f"  Seed={seed}  lr={lr}  batch_size={batch_size}  split={use_sentence_split}")
    print(f"{'=' * 60}")

    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- Load data --------------------------------------------------------
    train_raw, test_raw = load_seed(seed)

    if use_sentence_split:
        train_raw = expand_to_sentences(train_raw)
        test_raw = expand_to_sentences(test_raw)

    # ---- Validation split (§2.2) ------------------------------------------
    train_indices, val_indices = train_test_split(
        range(len(train_raw)),
        test_size=0.20,
        random_state=int_seed,
        stratify=[r["label"] for r in train_raw],
    )
    train_records = [train_raw[i] for i in train_indices]
    val_records = [train_raw[i] for i in val_indices]
    test_records = test_raw

    # ---- Tokeniser --------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

    train_ds = FOMCDataset(train_records, tokenizer)
    val_ds = FOMCDataset(val_records, tokenizer)
    test_ds = FOMCDataset(test_records, tokenizer)

    _g = torch.Generator()
    _g.manual_seed(int_seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=0, generator=_g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=collate_fn, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             collate_fn=collate_fn, num_workers=0)

    # ---- Model ------------------------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=3, hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        ignore_mismatched_sizes=True,
    )
    model.to(device)

    # ---- Optimiser + scheduler -------------------------------------------
    total_steps = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimiser, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    # ---- Training loop ----------------------------------------------------
    best_val_wf1 = -1.0
    best_epoch = 1
    patience_counter = 0
    history: list[dict] = []
    best_state = None
    train_start = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimiser, scheduler, device)
        val_loss, val_wf1, _, _, _ = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_loss": round(val_loss, 6),
            "val_weighted_f1": round(val_wf1, 6),
        })
        print(f"  Epoch {epoch:2d} | train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  val_wf1={val_wf1:.4f}")

        if val_wf1 > best_val_wf1:
            best_val_wf1 = val_wf1
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping at epoch {epoch}.")
                break

    train_time = time.time() - train_start

    # ---- Restore best checkpoint and save to disk -----------------------
    if best_state is not None:
        model.load_state_dict(best_state)
    model.save_pretrained(out_dir / "model")
    tokenizer.save_pretrained(out_dir / "model")

    # ---- Test evaluation -------------------------------------------------
    _, test_wf1, test_preds, test_true, test_probs = evaluate(model, test_loader, device)
    test_mf1 = f1_score(test_true, test_preds, average="macro", zero_division=0)
    test_acc = accuracy_score(test_true, test_preds)

    print(f"  >> Test weighted-F1={test_wf1:.4f}  macro-F1={test_mf1:.4f}  acc={test_acc:.4f}")

    # ---- Save outputs ----------------------------------------------------
    save_training_history(out_dir, history)
    save_learning_curve(out_dir, history, best_epoch)
    save_predictions(out_dir, test_records, test_preds, test_probs, seed, model_name)
    save_classification_report(out_dir, test_true, test_preds)
    save_confusion_matrices(out_dir, test_true, test_preds)

    # Per-class F1
    per_class = f1_score(test_true, test_preds, average=None, zero_division=0).tolist()

    metrics = {
        "seed": seed,
        "model_name": model_name,
        "best_epoch": best_epoch,
        "val_weighted_f1": round(best_val_wf1, 6),
        "test_weighted_f1": round(test_wf1, 6),
        "test_macro_f1": round(test_mf1, 6),
        "test_accuracy": round(test_acc, 6),
        "test_f1_dovish": round(per_class[0], 6),
        "test_f1_hawkish": round(per_class[1], 6),
        "test_f1_neutral": round(per_class[2], 6),
        "train_time_seconds": round(train_time, 2),
        "n_train": len(train_records),
        "n_val": len(val_records),
        "n_test": len(test_records),
    }
    save_metrics(out_dir, metrics)

    # Config
    import transformers, sklearn, torch as _torch
    cfg = {
        "model_name": model_name,
        "pretrained_checkpoint": CHECKPOINT,
        "checkpoint_revision": "main",
        "training_type": "full_fine_tuning",
        "tokenizer": CHECKPOINT,
        "max_length": MAX_LENGTH,
        "learning_rate": lr,
        "batch_size": batch_size,
        "epochs": MAX_EPOCHS,
        "best_epoch": best_epoch,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "random_seed": int_seed,
        "train_split_file": f"data/splits/{seed}_train.csv",
        "validation_split_file": f"data/splits/{seed}_val.csv",
        "test_split_file": f"FOMC_dataset_checkpoint/lab-manual-combine-test-{seed}.xlsx",
        "use_sentence_split": use_sentence_split,
        "library_versions": {
            "transformers": transformers.__version__,
            "torch": _torch.__version__,
            "sklearn": sklearn.__version__,
        },
        "hardware": str(device),
        "platform": platform.platform(),
    }
    save_config(out_dir, cfg)

    return metrics


## Hyperparameter search (seed 5768 only)


In [12]:
def hyperparameter_search(results_dir: Path, use_sentence_split: bool = False) -> tuple[float, int]:
    """Grid-search over HP_GRID on seed 5768. Returns best (lr, batch_size)."""
    device = get_device()
    print("\n=== Hyperparameter search on seed 5768 ===")
    best_lr, best_bs, best_wf1 = HP_GRID[0]["lr"], HP_GRID[0]["batch_size"], -1.0

    for hp in HP_GRID:
        lr, bs = hp["lr"], hp["batch_size"]
        tmp_dir = results_dir / f"_hpsearch_lr{lr}_bs{bs}"
        m = run_seed(
            "5768", tmp_dir, lr=lr, batch_size=bs,
            model_name="roberta_hpsearch",
            use_sentence_split=use_sentence_split,
            device=device,
        )
        if m["val_weighted_f1"] > best_wf1:
            best_wf1 = m["val_weighted_f1"]
            best_lr, best_bs = lr, bs

    print(f"\nBest HP: lr={best_lr}  batch_size={best_bs}  val_wf1={best_wf1:.4f}")
    return best_lr, best_bs


## Aggregate metrics


In [13]:
def aggregate(seed_metrics: list[dict], out_dir: Path, model_name: str):
    wf1s = [m["test_weighted_f1"] for m in seed_metrics]
    mf1s = [m["test_macro_f1"] for m in seed_metrics]
    accs = [m["test_accuracy"] for m in seed_metrics]

    agg = {
        "model_name": model_name,
        "seeds": SEEDS,
        "test_weighted_f1_mean": round(float(np.mean(wf1s)), 6),
        "test_weighted_f1_std": round(float(np.std(wf1s)), 6),
        "test_macro_f1_mean": round(float(np.mean(mf1s)), 6),
        "test_macro_f1_std": round(float(np.std(mf1s)), 6),
        "test_accuracy_mean": round(float(np.mean(accs)), 6),
        "test_accuracy_std": round(float(np.std(accs)), 6),
        "per_seed": seed_metrics,
    }
    with open(out_dir / "aggregate_metrics.json", "w") as f:
        json.dump(agg, f, indent=2)

    print(f"\n--- Aggregate ({model_name}) ---")
    print(f"  Weighted F1: {agg['test_weighted_f1_mean']:.4f} ± {agg['test_weighted_f1_std']:.4f}")
    print(f"  Macro    F1: {agg['test_macro_f1_mean']:.4f} ± {agg['test_macro_f1_std']:.4f}")
    print(f"  Accuracy:    {agg['test_accuracy_mean']:.4f} ± {agg['test_accuracy_std']:.4f}")


## Main


In [14]:
def main():
    parser = argparse.ArgumentParser(description="RoBERTa fine-tuning for FOMC sentiment")
    parser.add_argument(
        "--mode",
        choices=["combined", "combined_split", "both"],
        default="combined",
        help="Which experiment to run: combined (unsplit), combined_split (sentence-split), or both",
    )
    parser.add_argument(
        "--skip_hpsearch",
        action="store_true",
        help="Skip hyperparameter search and use defaults (lr=2e-5, batch_size=16)",
    )
    parser.add_argument("--lr", type=float, default=2e-5, help="Override learning rate")
    parser.add_argument("--batch_size", type=int, default=16, help="Override batch size")
    args = parser.parse_args()

    device = get_device()
    print(f"Device: {device}")

    modes = []
    if args.mode in ("combined", "both"):
        modes.append(("combined", False))
    if args.mode in ("combined_split", "both"):
        modes.append(("combined_split", True))

    for exp_name, use_split in modes:
        model_name = f"roberta_base_{exp_name}"
        results_dir = RESULTS_ROOT / f"roberta_base_{exp_name}"
        results_dir.mkdir(parents=True, exist_ok=True)

        # Step 1: hyperparameter search
        if args.skip_hpsearch:
            best_lr, best_bs = args.lr, args.batch_size
            print(f"Skipping HP search. Using lr={best_lr}, batch_size={best_bs}")
        else:
            best_lr, best_bs = hyperparameter_search(results_dir, use_sentence_split=use_split)

        # Step 2: run all three seeds with frozen hyperparameters
        seed_metrics = []
        for seed in SEEDS:
            seed_dir = results_dir / f"seed_{seed}"
            m = run_seed(
                seed, seed_dir,
                lr=best_lr, batch_size=best_bs,
                model_name=model_name,
                use_sentence_split=use_split,
                device=device,
            )
            seed_metrics.append(m)

        # Step 3: aggregate
        aggregate(seed_metrics, results_dir, model_name)


## Experiment configuration

Set the experiment options below. `combined` runs the standard model, while `combined_split` runs the sentence-split extension.


In [15]:
MODE = "combined"  # "combined", "combined_split", or "both"
SKIP_HPSEARCH = True
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
RUN_SEEDS = SEEDS

print(f"Device: {get_device()}")
print(f"Seeds: {RUN_SEEDS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size: {BATCH_SIZE}")

Device: cpu
Seeds: ['5768', '78516', '944601']
Learning rate: 2e-05
Batch size: 16


## Dataset check

Load the first configured split and display its sizes and class distributions before training.


In [16]:
preview_train, preview_test = load_seed(RUN_SEEDS[0])
print(f"Seed {RUN_SEEDS[0]}")
print(f"Training records: {len(preview_train)}")
print(f"Test records: {len(preview_test)}")
print(f"Training labels: {dict(sorted(Counter(r['label'] for r in preview_train).items()))}")
print(f"Test labels: {dict(sorted(Counter(r['label'] for r in preview_test).items()))}")

Seed 5768
Training records: 1903
Test records: 476
Training labels: {0: 498, 1: 476, 2: 929}
Test labels: {0: 127, 1: 124, 2: 225}


## Train and evaluate

This cell logs each epoch, evaluates every configured seed, saves per-seed outputs, and writes aggregate metrics.


In [ ]:
device = get_device()
results_dir = RESULTS_ROOT / "roberta"
results_dir.mkdir(parents=True, exist_ok=True)

if SKIP_HPSEARCH:
    best_lr, best_batch_size = LEARNING_RATE, BATCH_SIZE
    print(f"Skipping HP search. Using lr={best_lr}, batch_size={best_batch_size}")
else:
    best_lr, best_batch_size = hyperparameter_search(results_dir)

seed_metrics = []
for seed in RUN_SEEDS:
    metrics = run_seed(
        seed=seed,
        out_dir=results_dir / f"seed_{seed}",
        lr=best_lr,
        batch_size=best_batch_size,
        model_name="roberta",
        use_sentence_split=False,
        device=device,
    )
    seed_metrics.append(metrics)

aggregate(seed_metrics, results_dir, "roberta")
completed_results = {"roberta": results_dir}

Skipping HP search. Using lr=2e-05, batch_size=16

  Seed=5768  lr=2e-05  batch_size=16  split=False


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4676.00it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Overall training:  10%|█         | 96/960 [13:19<1:59:56,  8.33s/batch, epoch_avg=1.0449, loss=0.5744]


  Epoch  1 | train_loss=1.0449  val_loss=0.9181  val_wf1=0.4148


Overall training:  20%|██        | 192/960 [13:36<1:48:52,  8.51s/batch, epoch_avg=0.9043, loss=1.0639]


  Epoch  2 | train_loss=0.9043  val_loss=0.8151  val_wf1=0.6079


...

  Epoch  7 | train_loss=0.1292  val_loss=1.2207  val_wf1=0.6951


Overall training:  80%|████████  | 768/960 [08:34<17:09,  5.36s/batch, epoch_avg=0.0693, loss=0.0019]


  Epoch  8 | train_loss=0.0693  val_loss=1.4359  val_wf1=0.6983
  Early stopping at epoch 8.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]


  >> Test weighted-F1=0.6959  macro-F1=0.6839  acc=0.6933

--- Aggregate (roberta_base_combined) ---
  Weighted F1: 0.6887 ± 0.0148
  Macro    F1: 0.6739 ± 0.0148
  Accuracy:    0.6870 ± 0.0152


## Accuracy summary

Read and display the aggregate accuracy and F1 scores saved by the training cell.


In [ ]:
for experiment_name, results_dir in completed_results.items():
    metrics_path = results_dir / "aggregate_metrics.json"
    with metrics_path.open(encoding="utf-8") as file:
        result = json.load(file)

    print(f"\n{result['model_name']}")
    print(f"Accuracy:    {result['test_accuracy_mean']:.4f} ± {result['test_accuracy_std']:.4f}")
    print(f"Weighted F1: {result['test_weighted_f1_mean']:.4f} ± {result['test_weighted_f1_std']:.4f}")
    print(f"Macro F1:    {result['test_macro_f1_mean']:.4f} ± {result['test_macro_f1_std']:.4f}")

## RoBERTa experiment matrix

The base **RoBERTa** experiment is the earlier **Train and evaluate** cell and uses original training data. Run the three setup/preflight cells below once, then run each augmented experiment independently.

All augmented experiments train with augmented rows, validate on original rows from held-out source families, and evaluate on the original unaugmented test set. Progress bars show percentage and ETA. Reusable weights are saved under `9444/results/<experiment>/seed_<seed>/model`.

In [24]:
import random
from dataclasses import asdict, dataclass

import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

VARIANT_SEEDS = list(RUN_SEEDS)
VARIANT_HP_GRID = [
    {"learning_rate": 1e-5, "batch_size": 8},
    {"learning_rate": 2e-5, "batch_size": 8},
    {"learning_rate": 5e-5, "batch_size": 8},
    {"learning_rate": 1e-5, "batch_size": 16},
    {"learning_rate": 2e-5, "batch_size": 16},
    {"learning_rate": 5e-5, "batch_size": 16},
]


@dataclass(frozen=True)
class NotebookVariant:
    name: str
    use_dapt: bool = False
    use_average_weighted_hp_search: bool = False
    use_class_balancing: bool = False
    label_smoothing: float = 0.0


class DAPTTextDataset(Dataset):
    def __init__(self, records, tokenizer):
        self.input_ids = tokenizer(
            [normalise(record["sentence"]) for record in records],
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
        )["input_ids"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return {"input_ids": self.input_ids[index]}


def set_variant_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def variant_class_weights(records, device):
    counts = Counter(record["label"] for record in records)
    total = len(records)
    weights = [total / (len(LABEL_NAMES) * counts[label]) for label in range(len(LABEL_NAMES))]
    return torch.tensor(weights, dtype=torch.float32, device=device)


def variant_loss(logits, labels, class_weights, label_smoothing):
    return F.cross_entropy(
        logits,
        labels,
        weight=class_weights,
        label_smoothing=label_smoothing,
    )


@torch.no_grad()
def evaluate_variant(model, loader, device, class_weights, label_smoothing):
    model.eval()
    losses, predictions, labels, probabilities = [], [], [], []
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits
        loss = variant_loss(logits, batch["labels"], class_weights, label_smoothing)
        probs = torch.softmax(logits, dim=-1)
        losses.append(loss.item())
        predictions.extend(probs.argmax(dim=-1).cpu().tolist())
        labels.extend(batch["labels"].cpu().tolist())
        probabilities.extend(probs.cpu().tolist())
    return {
        "loss": float(np.mean(losses)),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "accuracy": accuracy_score(labels, predictions),
        "predictions": predictions,
        "labels": labels,
        "probabilities": probabilities,
    }


def run_notebook_dapt(records, output_dir, seed, device):
    if (output_dir / "config.json").exists():
        print(f"Reusing cached DAPT checkpoint: {output_dir}")
        return output_dir

    set_variant_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
    dataset = DAPTTextDataset(records, tokenizer)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collator)
    model = AutoModelForMaskedLM.from_pretrained(CHECKPOINT).to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=WEIGHT_DECAY)
    dapt_epochs = 3
    total_steps = max(len(loader) * dapt_epochs, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    model.train()
    with tqdm(total=total_steps, desc=f"DAPT seed={seed}", unit="batch", dynamic_ncols=True) as progress:
        for epoch in range(1, dapt_epochs + 1):
            epoch_losses = []
            for batch in loader:
                batch = {key: value.to(device) for key, value in batch.items()}
                optimiser.zero_grad()
                loss = model(**batch).loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                scheduler.step()
                epoch_losses.append(loss.item())
                progress.update()
                progress.set_postfix(epoch=f"{epoch}/{dapt_epochs}", mlm_loss=f"{loss.item():.4f}")
            progress.write(f"DAPT epoch {epoch}: MLM loss={np.mean(epoch_losses):.4f}")

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output_dir


def train_notebook_variant_seed(
    seed,
    variant,
    learning_rate,
    batch_size,
    device,
    save_outputs=True,
):
    integer_seed = int(seed)
    set_variant_seed(integer_seed)
    all_train_records, test_records = load_seed(seed)
    train_indices, validation_indices = train_test_split(
        range(len(all_train_records)),
        test_size=0.2,
        random_state=integer_seed,
        stratify=[record["label"] for record in all_train_records],
    )
    train_records = [all_train_records[index] for index in train_indices]
    validation_records = [all_train_records[index] for index in validation_indices]

    output_dir = RESULTS_ROOT / variant.name / f"seed_{seed}"
    checkpoint = CHECKPOINT
    if variant.use_dapt:
        checkpoint = run_notebook_dapt(
            train_records,
            RESULTS_ROOT / variant.name / "_dapt" / f"seed_{seed}",
            integer_seed,
            device,
        )

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    generator = torch.Generator().manual_seed(integer_seed)
    train_loader = DataLoader(
        FOMCDataset(train_records, tokenizer),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        collate_fn=collate_fn,
        num_workers=0,
    )
    validation_loader = DataLoader(
        FOMCDataset(validation_records, tokenizer),
        batch_size=batch_size,
        collate_fn=collate_fn,
        num_workers=0,
    )
    test_loader = DataLoader(
        FOMCDataset(test_records, tokenizer),
        batch_size=batch_size,
        collate_fn=collate_fn,
        num_workers=0,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(LABEL_NAMES),
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        ignore_mismatched_sizes=True,
    ).to(device)
    class_weights = variant_class_weights(train_records, device) if variant.use_class_balancing else None
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    total_steps = max(len(train_loader) * MAX_EPOCHS, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    best_score, best_epoch, patience = -1.0, 0, 0
    best_state = None
    history = []
    started = time.time()
    with tqdm(
        total=total_steps,
        desc=f"{variant.name} seed={seed}",
        unit="batch",
        dynamic_ncols=True,
    ) as progress:
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []
            for batch in train_loader:
                batch = {key: value.to(device) for key, value in batch.items()}
                optimiser.zero_grad()
                logits = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                ).logits
                loss = variant_loss(logits, batch["labels"], class_weights, variant.label_smoothing)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                scheduler.step()
                train_losses.append(loss.item())
                progress.update()
                progress.set_postfix(epoch=f"{epoch}/{MAX_EPOCHS}", loss=f"{loss.item():.4f}")

            validation = evaluate_variant(
                model,
                validation_loader,
                device,
                class_weights,
                variant.label_smoothing,
            )
            epoch_metrics = {
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "val_loss": validation["loss"],
                "val_weighted_f1": validation["weighted_f1"],
            }
            history.append(epoch_metrics)
            progress.set_postfix(
                epoch=f"{epoch}/{MAX_EPOCHS}",
                loss=f"{epoch_metrics['train_loss']:.4f}",
                val_f1=f"{validation['weighted_f1']:.4f}",
            )
            progress.write(
                f"{variant.name} seed={seed} epoch={epoch}: "
                f"train_loss={epoch_metrics['train_loss']:.4f}, "
                f"val_loss={validation['loss']:.4f}, "
                f"val_weighted_f1={validation['weighted_f1']:.4f}"
            )
            if validation["weighted_f1"] > best_score:
                best_score = validation["weighted_f1"]
                best_epoch = epoch
                best_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= EARLY_STOPPING_PATIENCE:
                    progress.write(f"Early stopping after epoch {epoch}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    metrics = {
        "seed": seed,
        "variant": variant.name,
        "validation_weighted_f1": best_score,
        "best_epoch": best_epoch,
        "training_seconds": time.time() - started,
    }
    if save_outputs:
        test = evaluate_variant(model, test_loader, device, class_weights, variant.label_smoothing)
        metrics.update({
            "test_weighted_f1": test["weighted_f1"],
            "test_macro_f1": test["macro_f1"],
            "test_accuracy": test["accuracy"],
        })
        output_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(output_dir / "model")
        tokenizer.save_pretrained(output_dir / "model")
        save_metrics(output_dir, metrics)
        save_training_history(output_dir, history)
        save_learning_curve(output_dir, history, best_epoch)
        save_predictions(
            output_dir,
            test_records,
            test["predictions"],
            test["probabilities"],
            seed,
            variant.name,
        )
        save_classification_report(output_dir, test["labels"], test["predictions"])
        save_confusion_matrices(output_dir, test["labels"], test["predictions"])
        save_config(output_dir, {
            "variant": asdict(variant),
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "checkpoint": str(checkpoint),
            "class_weights": class_weights.cpu().tolist() if class_weights is not None else None,
        })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


def average_weighted_notebook_search(variant, device):
    results_dir = RESULTS_ROOT / variant.name
    trials = []
    for hyperparameters in tqdm(
        VARIANT_HP_GRID,
        desc=f"{variant.name} HP search",
        unit="trial",
        dynamic_ncols=True,
    ):
        scores = []
        for seed in VARIANT_SEEDS:
            metrics = train_notebook_variant_seed(
                seed,
                variant,
                hyperparameters["learning_rate"],
                hyperparameters["batch_size"],
                device,
                save_outputs=False,
            )
            scores.append(metrics["validation_weighted_f1"])
        trial = {
            **hyperparameters,
            "per_seed_validation_weighted_f1": dict(zip(VARIANT_SEEDS, scores)),
            "mean_validation_weighted_f1": float(np.mean(scores)),
            "std_validation_weighted_f1": float(np.std(scores)),
        }
        trials.append(trial)
        print(f"HP trial complete: {trial}")

    results_dir.mkdir(parents=True, exist_ok=True)
    with (results_dir / "hyperparameter_search.json").open("w", encoding="utf-8") as file:
        json.dump(trials, file, indent=2)
    best = max(trials, key=lambda trial: trial["mean_validation_weighted_f1"])
    print(f"Best hyperparameters: {best}")
    return best["learning_rate"], best["batch_size"]


def run_notebook_variant(variant):
    device = get_device()
    learning_rate, batch_size = LEARNING_RATE, BATCH_SIZE
    print(f"Variant: {variant.name}")
    print(f"Device: {device}; seeds: {VARIANT_SEEDS}")
    if variant.use_average_weighted_hp_search:
        learning_rate, batch_size = average_weighted_notebook_search(variant, device)

    seed_metrics = []
    for seed in tqdm(VARIANT_SEEDS, desc=f"{variant.name} final seeds", unit="seed"):
        seed_metrics.append(
            train_notebook_variant_seed(seed, variant, learning_rate, batch_size, device)
        )

    aggregate_metrics = {
        "variant": variant.name,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "seeds": VARIANT_SEEDS,
        "test_weighted_f1_mean": float(np.mean([item["test_weighted_f1"] for item in seed_metrics])),
        "test_weighted_f1_std": float(np.std([item["test_weighted_f1"] for item in seed_metrics])),
        "test_macro_f1_mean": float(np.mean([item["test_macro_f1"] for item in seed_metrics])),
        "test_accuracy_mean": float(np.mean([item["test_accuracy"] for item in seed_metrics])),
        "per_seed": seed_metrics,
    }
    results_dir = RESULTS_ROOT / variant.name
    results_dir.mkdir(parents=True, exist_ok=True)
    with (results_dir / "aggregate_metrics.json").open("w", encoding="utf-8") as file:
        json.dump(aggregate_metrics, file, indent=2)
    print(json.dumps(aggregate_metrics, indent=2))
    return aggregate_metrics


print(f"Native variant training ready on {get_device()} for seeds {VARIANT_SEEDS}")

Native variant training ready on cpu for seeds ['5768', '78516', '944601']


In [25]:
# Augmented-data protocol: augmented train, original-only validation and test.
AUGMENTED_TRAIN_DIR = (
    ROOT.parent
    / "dataset"
    / "test-and-training"
    / "augmented_data"
    / "augmented_train_data"
)


def load_augmented_seed(seed: str) -> tuple[list[dict], list[dict]]:
    train_path = AUGMENTED_TRAIN_DIR / f"lab-manual-combine-train-{seed}.xlsx"
    if not train_path.exists():
        raise FileNotFoundError(f"Augmented training file not found: {train_path}")
    augmented_train = _read_xlsx_bytes(train_path.read_bytes())
    _, original_test = load_seed(seed)
    return augmented_train, original_test


def split_augmented_train_validation(records: list[dict], seed: int):
    original_records = [record for record in records if int(record.get("augmented", 0)) == 0]
    train_indices, validation_indices = train_test_split(
        range(len(original_records)),
        test_size=0.2,
        random_state=seed,
        stratify=[record["label"] for record in original_records],
    )
    training_source_ids = {original_records[index]["index"] for index in train_indices}
    train_records = [record for record in records if record["index"] in training_source_ids]
    validation_records = [original_records[index] for index in validation_indices]
    return train_records, validation_records


@dataclass(frozen=True)
class NotebookVariant:
    name: str
    use_augmented_data: bool = False
    use_dapt: bool = False
    use_average_weighted_hp_search: bool = False
    use_class_balancing: bool = False
    label_smoothing: float = 0.0


def train_notebook_variant_seed(
    seed,
    variant,
    learning_rate,
    batch_size,
    device,
    save_outputs=True,
):
    integer_seed = int(seed)
    set_variant_seed(integer_seed)

    if variant.use_augmented_data:
        all_train_records, test_records = load_augmented_seed(seed)
        train_records, validation_records = split_augmented_train_validation(
            all_train_records,
            integer_seed,
        )
    else:
        all_train_records, test_records = load_seed(seed)
        train_indices, validation_indices = train_test_split(
            range(len(all_train_records)),
            test_size=0.2,
            random_state=integer_seed,
            stratify=[record["label"] for record in all_train_records],
        )
        train_records = [all_train_records[index] for index in train_indices]
        validation_records = [all_train_records[index] for index in validation_indices]

    print(
        f"{variant.name} seed={seed}: train={len(train_records)}, "
        f"validation={len(validation_records)}, test={len(test_records)}"
    )
    output_dir = RESULTS_ROOT / variant.name / f"seed_{seed}"
    checkpoint = CHECKPOINT
    if variant.use_dapt:
        checkpoint = run_notebook_dapt(
            train_records,
            RESULTS_ROOT / variant.name / "_dapt" / f"seed_{seed}",
            integer_seed,
            device,
        )

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    generator = torch.Generator().manual_seed(integer_seed)
    train_loader = DataLoader(
        FOMCDataset(train_records, tokenizer),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        collate_fn=collate_fn,
        num_workers=0,
    )
    validation_loader = DataLoader(
        FOMCDataset(validation_records, tokenizer),
        batch_size=batch_size,
        collate_fn=collate_fn,
        num_workers=0,
    )
    test_loader = DataLoader(
        FOMCDataset(test_records, tokenizer),
        batch_size=batch_size,
        collate_fn=collate_fn,
        num_workers=0,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(LABEL_NAMES),
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        ignore_mismatched_sizes=True,
    ).to(device)
    class_weights = (
        variant_class_weights(train_records, device)
        if variant.use_class_balancing
        else None
    )
    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = max(len(train_loader) * MAX_EPOCHS, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    best_score, best_epoch, patience = -1.0, 0, 0
    best_state = None
    history = []
    started = time.time()
    with tqdm(
        total=total_steps,
        desc=f"{variant.name} seed={seed}",
        unit="batch",
        dynamic_ncols=True,
    ) as progress:
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []
            for batch in train_loader:
                batch = {key: value.to(device) for key, value in batch.items()}
                optimiser.zero_grad()
                logits = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                ).logits
                loss = variant_loss(
                    logits,
                    batch["labels"],
                    class_weights,
                    variant.label_smoothing,
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                scheduler.step()
                train_losses.append(loss.item())
                progress.update()
                progress.set_postfix(
                    epoch=f"{epoch}/{MAX_EPOCHS}",
                    loss=f"{loss.item():.4f}",
                )

            validation = evaluate_variant(
                model,
                validation_loader,
                device,
                class_weights,
                variant.label_smoothing,
            )
            epoch_metrics = {
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "val_loss": validation["loss"],
                "val_weighted_f1": validation["weighted_f1"],
            }
            history.append(epoch_metrics)
            progress.set_postfix(
                epoch=f"{epoch}/{MAX_EPOCHS}",
                loss=f"{epoch_metrics['train_loss']:.4f}",
                val_f1=f"{validation['weighted_f1']:.4f}",
            )
            progress.write(
                f"{variant.name} seed={seed} epoch={epoch}: "
                f"train_loss={epoch_metrics['train_loss']:.4f}, "
                f"val_loss={validation['loss']:.4f}, "
                f"val_weighted_f1={validation['weighted_f1']:.4f}"
            )
            if validation["weighted_f1"] > best_score:
                best_score = validation["weighted_f1"]
                best_epoch = epoch
                best_state = {
                    key: value.cpu().clone()
                    for key, value in model.state_dict().items()
                }
                patience = 0
            else:
                patience += 1
                if patience >= EARLY_STOPPING_PATIENCE:
                    progress.write(f"Early stopping after epoch {epoch}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    metrics = {
        "seed": seed,
        "variant": variant.name,
        "validation_weighted_f1": best_score,
        "best_epoch": best_epoch,
        "training_seconds": time.time() - started,
    }
    if save_outputs:
        test = evaluate_variant(
            model,
            test_loader,
            device,
            class_weights,
            variant.label_smoothing,
        )
        metrics.update({
            "test_weighted_f1": test["weighted_f1"],
            "test_macro_f1": test["macro_f1"],
            "test_accuracy": test["accuracy"],
        })
        print(
            f"{variant.name} seed={seed} test: "
            f"weighted_f1={test['weighted_f1']:.4f}, "
            f"macro_f1={test['macro_f1']:.4f}, "
            f"accuracy={test['accuracy']:.4f}"
        )
        output_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(output_dir / "model")
        tokenizer.save_pretrained(output_dir / "model")
        save_metrics(output_dir, metrics)
        save_training_history(output_dir, history)
        save_learning_curve(output_dir, history, best_epoch)
        save_predictions(
            output_dir,
            test_records,
            test["predictions"],
            test["probabilities"],
            seed,
            variant.name,
        )
        save_classification_report(
            output_dir,
            test["labels"],
            test["predictions"],
        )
        save_confusion_matrices(
            output_dir,
            test["labels"],
            test["predictions"],
        )
        save_config(output_dir, {
            "variant": asdict(variant),
            "data_source": (
                str(AUGMENTED_TRAIN_DIR)
                if variant.use_augmented_data
                else str(DATA_ZIP)
            ),
            "validation_data": "original rows only",
            "test_data": "original unaugmented test split",
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "checkpoint": str(checkpoint),
            "class_weights": (
                class_weights.cpu().tolist()
                if class_weights is not None
                else None
            ),
        })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


print(f"Augmented-data variants ready: {AUGMENTED_TRAIN_DIR}")

Augmented-data variants ready: C:\Users\Anthony.Ivanov\Downloads\COMP9444_26T2_FOMC_Analysis-main\COMP9444_26T2_FOMC_Analysis-main\dataset\test-and-training\augmented_data\augmented_train_data


In [28]:
# Enforce augmentation-family isolation, then verify every seed before training.
from sklearn.model_selection import StratifiedGroupKFold


def augmentation_family(record: dict) -> str:
    # Every generated row preserves these source fields, including label-flipping swaps.
    return f"{record['index']}\x1f{record['year']}"


def split_augmented_train_validation(records: list[dict], seed: int):
    original_records = [
        record for record in records
        if int(record.get("augmented", 0)) == 0
    ]
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    train_indices, validation_indices = next(
        splitter.split(
            original_records,
            [record["label"] for record in original_records],
            groups=[augmentation_family(record) for record in original_records],
        )
    )
    training_families = {
        augmentation_family(original_records[index]) for index in train_indices
    }
    train_records = [
        record for record in records
        if augmentation_family(record) in training_families
    ]
    validation_records = [original_records[index] for index in validation_indices]
    return train_records, validation_records


for seed in VARIANT_SEEDS:
    augmented_records, original_test_records = load_augmented_seed(seed)
    augmented_train_records, original_validation_records = split_augmented_train_validation(
        augmented_records,
        int(seed),
    )
    train_families = {augmentation_family(record) for record in augmented_train_records}
    validation_families = {
        augmentation_family(record) for record in original_validation_records
    }
    assert train_families.isdisjoint(validation_families)
    assert all(int(record.get("augmented", 0)) == 0 for record in original_validation_records)
    assert all("augmented" not in record for record in original_test_records)
    print(
        f"Seed {seed}: augmented train={len(augmented_train_records)}, "
        f"original validation={len(original_validation_records)}, "
        f"original test={len(original_test_records)}"
    )

Seed 5768: augmented train=4146, original validation=371, original test=476
Seed 78516: augmented train=4148, original validation=371, original test=476
Seed 944601: augmented train=4118, original validation=372, original test=476


In [ ]:
# RoBERTa + Augmented Data
augmented_data_results = run_notebook_variant(
    NotebookVariant(
        name="roberta_augmented_data",
        use_augmented_data=True,
    )
)

In [ ]:
# RoBERTa + AD + DAPT
dapt_results = run_notebook_variant(
    NotebookVariant(
        name="roberta_ad_dapt",
        use_augmented_data=True,
        use_dapt=True,
    )
)

In [ ]:
# RoBERTa + AD + Class Balancing
class_balancing_results = run_notebook_variant(
    NotebookVariant(
        name="roberta_ad_class_balancing",
        use_augmented_data=True,
        use_class_balancing=True,
    )
)

In [ ]:
# RoBERTa + AD + Label Smoothing
label_smoothing_results = run_notebook_variant(
    NotebookVariant(
        name="roberta_ad_label_smoothing",
        use_augmented_data=True,
        label_smoothing=0.1,
    )
)

In [ ]:
# RoBERTa + AD + Weighted Averaging
# Selects hyperparameters by mean validation weighted-F1 across all seeds.
weighted_averaging_results = run_notebook_variant(
    NotebookVariant(
        name="roberta_ad_weighted_averaging",
        use_augmented_data=True,
        use_average_weighted_hp_search=True,
    )
)

In [31]:
# RoBERTa + All
# Combines augmented data, DAPT, weighted-averaging HP selection,
# class balancing, and label smoothing.
all_enhancements_results = run_notebook_variant(
    NotebookVariant(
        name="roberta_all",
        use_augmented_data=True,
        use_dapt=True,
        use_average_weighted_hp_search=True,
        use_class_balancing=True,
        label_smoothing=0.1,
    )
)

Variant: roberta_all
Device: cpu; seeds: ['5768', '78516', '944601']


roberta_all HP search:   0%|          | 0/6 [00:00<?, ?trial/s]

roberta_all seed=5768: train=4146, validation=371, test=476
Reusing cached DAPT checkpoint: C:\Users\Anthony.Ivanov\Downloads\COMP9444_26T2_FOMC_Analysis-main\COMP9444_26T2_FOMC_Analysis-main\9444\results\roberta_all\_dapt\seed_5768


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5551.75it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: C:\Users\Anthony.Ivanov\Downloads\COMP9444_26T2_FOMC_Analysis-main\COMP9444_26T2_FOMC_Analysis-main\9444\results\roberta_all\_dapt\seed_5768
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

ro

roberta_all seed=5768 epoch=1: train_loss=1.0560, val_loss=0.8763, val_weighted_f1=0.6547


...

roberta_all seed=944601 epoch=3: train_loss=0.5520, val_loss=1.0516, val_weighted_f1=0.6794



roberta_all seed=944601:  40%|████      | 2060/5150 [2:18:41<3:28:01,  4.04s/batch, epoch=4/10, loss=0.4340, val_f1=0.6939]


roberta_all seed=944601 epoch=4: train_loss=0.4340, val_loss=1.0753, val_weighted_f1=0.6939
Early stopping after epoch 4
roberta_all seed=944601 test: weighted_f1=0.6624, macro_f1=0.6529, accuracy=0.6597


roberta_all final seeds: 100%|██████████| 3/3 [5:49:44<00:00, 6994.95s/seed]  

{
  "variant": "roberta_all",
  "learning_rate": 1e-05,
  "batch_size": 8,
  "seeds": [
    "5768",
    "78516",
    "944601"
  ],
  "test_weighted_f1_mean": 0.6677071077084533,
  "test_weighted_f1_std": 0.015929949443833927,
  "test_macro_f1_mean": 0.6541667579740308,
  "test_accuracy_mean": 0.6659663865546218,
  "per_seed": [
    {
      "seed": "5768",
      "variant": "roberta_all",
      "validation_weighted_f1": 0.719801809229708,
      "best_epoch": 2,
      "training_seconds": 4892.959595441818,
      "test_weighted_f1": 0.65139225546678,
      "test_macro_f1": 0.6324286094022936,
      "test_accuracy": 0.6533613445378151
    },
    {
      "seed": "78516",
      "variant": "roberta_all",
      "validation_weighted_f1": 0.6785266201216162,
      "best_epoch": 2,
      "training_seconds": 7646.229595661163,
      "test_weighted_f1": 0.6893186830470567,
      "test_macro_f1": 0.6771476072946662,
      "test_accuracy": 0.6848739495798319
    },
    {
      "seed": "944601",
      